In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

def fetch_page(session, base_url, service_key, bgn_dt, end_dt, page_no, num_of_rows):
    """단일 페이지 요청 함수 (Session 활용)"""
    url = (
        f"{base_url}?serviceKey={service_key}"
        f"&pageNo={page_no}&numOfRows={num_of_rows}&type=json&inqryDiv=1"
        f"&inqryBgnDt={bgn_dt}&inqryEndDt={end_dt}"
    )
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

def fetch_all_items_for_period(session, base_url, service_key, bgn_dt, end_dt, num_of_rows=500):
    """특정 기간에 대해 페이지네이션으로 전체 아이템 수집 (진행상황 표시 추가)"""
    all_items = []
    page_no = 1
    
    # 한 줄에 이어서 진행 상황을 출력하도록 설정
    print("    ↳ 세부 데이터 다운로드: ", end="", flush=True)

    while True:
        try:
            data = fetch_page(session, base_url, service_key, bgn_dt, end_dt, page_no, num_of_rows)
            body = data.get('response', {}).get('body', {})
            items = body.get('items', [])
            total_count = body.get('totalCount', 0)

            if not items:
                break

            all_items.extend(items)
            # 현재 다운로드 중인 페이지와 누적 건수를 실시간으로 화면에 찍어줌
            print(f"[{page_no}p({len(all_items)}건)] ", end="", flush=True)

            if len(all_items) >= total_count:
                break

            page_no += 1
            time.sleep(0.1) # 대기 시간 최소화
            
        except Exception as e:
            print(f"\n    ⚠️ API 서버 지연 에러 (페이지 {page_no}): {e}")
            break

    print(" ✔️ 완료")
    return all_items


def get_itcen_group_bids(days_back=30, chunk_days=5): # chunk_days를 5일로 조정
    """아이티센 그룹 맞춤형 공고 및 필수 문서(제안+입찰) 동시 포함 필터링"""
    SERVICE_KEY = "18dbc10d66b4f8764dcc5ada60ebe8550b7232c4a4883480afd6c24d8c5aed78"
    BASE_URL = "https://apis.data.go.kr/1230000/ad/BidPublicInfoService/getBidPblancListInfoServc"

    today = datetime.now()
    start_date = today - timedelta(days=days_back)

    CLOIT_CODES = ['81111513', '81111595', '81111596', '81111594', '81111599', '81111598', '81112002', '80101507', '80101698']
    ENTEC_CODES = ['81111799', '81111801', '81111809', '81111708', '81151699']
    GLOBAL_CODES = ['81111899', '81112399', '81112299', '81111811', '81112199', '80141619']
    ALL_TARGET_CODES = CLOIT_CODES + ENTEC_CODES + GLOBAL_CODES

    print("⏳ [로딩 중] 조달청 API에서 공고 데이터를 수집 및 분석하고 있습니다...\n")

    periods = []
    cursor = start_date
    while cursor < today:
        chunk_end = min(cursor + timedelta(days=chunk_days), today)
        bgn_dt = cursor.strftime("%Y%m%d0000")
        end_dt = chunk_end.strftime("%Y%m%d2359")
        periods.append((bgn_dt, end_dt))
        cursor = chunk_end

    raw_items = []
    
    with requests.Session() as session:
        try:
            for idx, (bgn_dt, end_dt) in enumerate(periods, 1):
                print(f"  📅 기간 조회 중 ({idx}/{len(periods)}): {bgn_dt} ~ {end_dt}")
                period_items = fetch_all_items_for_period(session, BASE_URL, SERVICE_KEY, bgn_dt, end_dt)
                raw_items.extend(period_items)
                time.sleep(0.5) # 서버 블락(Block)을 피하기 위해 청크 사이 약간의 휴식
        except Exception as e:
            print(f"❌ 데이터 수집 에러 발생: {str(e)}")
            return []

    # 중복 제거
    seen = set()
    deduped_items = []
    for item in raw_items:
        key = item.get('bidNtceNo')
        if key not in seen:
            seen.add(key)
            deduped_items.append(item)

    filtered_results = []
    for item in deduped_items:
        clsfc_no = item.get('pubPrcrmntClsfcNo')

        if clsfc_no in ALL_TARGET_CODES:
            target_company = "☁️ 클로잇 (CloIT)" if clsfc_no in CLOIT_CODES else ("🌐 엔텍 (NTEC)" if clsfc_no in ENTEC_CODES else "🏢 글로벌 (Global)")

            has_proposal = False # 제안요청서 포함 여부
            has_bid_notice = False # 입찰공고문 포함 여부
            temp_files = []

            for i in range(1, 21):
                file_nm = item.get(f"ntceSpecFileNm{i}", "")
                file_url = item.get(f"ntceSpecDocUrl{i}", "")

                if file_nm:
                    if "제안" in file_nm:
                        has_proposal = True
                    if "입찰" in file_nm:
                        has_bid_notice = True
                    
                    if any(keyword in file_nm for keyword in ["제안", "과업", "입찰"]):
                        temp_files.append({"fileName": file_nm, "downloadUrl": file_url})

            # 제안요청서와 입찰공고문이 '모두' 존재하는 경우만 최종 결과에 추가
            if has_proposal and has_bid_notice:
                bid_info = {
                    "targetCompany": target_company,
                    "bidNtceNo": item.get('bidNtceNo'),
                    "bidNtceNm": item.get('bidNtceNm'),
                    "ntceInsttNm": item.get('ntceInsttNm'),
                    "pubPrcrmntClsfcNm": item.get('pubPrcrmntClsfcNm'),
                    "bidClseDt": item.get('bidClseDt'),
                    "asignBdgtAmt": item.get('asignBdgtAmt'),
                    "bidNtceDtlUrl": item.get('bidNtceDtlUrl'),
                    "targetFiles": temp_files
                }
                filtered_results.append(bid_info)

    return filtered_results

def save_to_excel(results):
    """추출된 리스트를 바탕으로 엑셀 파일 생성"""
    if not results:
        return

    excel_data = []

    for r in results:
        budget = f"{int(r['asignBdgtAmt']):,}원" if r['asignBdgtAmt'] else "미정"

        files_text = ""
        for idx, f in enumerate(r['targetFiles'], 1):
            files_text += f"{idx}. {f['fileName']}\n🔗 {f['downloadUrl']}\n\n"

        if not files_text:
            files_text = "타겟 첨부파일 없음"

        excel_data.append({
            "타겟 그룹사": r['targetCompany'],
            "공고명": r['bidNtceNm'],
            "세부분류명": r['pubPrcrmntClsfcNm'],
            "수요기관": r['ntceInsttNm'],
            "예산": budget,
            "마감일시": r['bidClseDt'],
            "공고 상세링크": r['bidNtceDtlUrl'],
            "첨부파일 다운로드": files_text.strip()
        })

    df = pd.DataFrame(excel_data)
    file_name = f"아이티센_그룹_맞춤형_공고_{datetime.now().strftime('%Y%m%d')}.csv"
    df.to_csv(file_name, index=False, encoding='utf-8-sig')
    print(f"\n💾 CSV 저장 완료: [{file_name}] 파일이 현재 폴더에 생성됨.")

if __name__ == "__main__":
    results = get_itcen_group_bids(days_back=7, chunk_days=1)

    if not results:
        print("조건에 맞는 타겟 공고가 없거나 조회에 실패함.")
    else:
        save_to_excel(results)
        
        print(f"✅ 총 {len(results)}건의 맞춤형 공고가 검색됨.\n")
        print("=" * 60)
